# TP N°2 — CamemBERT (phrases) vs Sentence‑BERT : comprendre et comparer les sentence embeddings

## 🎯 Objectifs d'apprentissage
- Extraire des **embeddings de tokens** et **de phrases** avec CamemBERT (CLS, Mean).
- Obtenir des **embeddings de phrases** avec Sentence‑BERT.
- Comparer les similarités de phrases (proches vs éloignées) et constater la supériorité de SBERT.


Ce notebook est organisé en plusieurs parties :

- **Section 0 — Rappels NumPy et PyTorch**  
  Rappels sur les tableaux, les tenseurs et leurs dimensions. Cette section peut être consultée selon les besoins.

- **Partie A — CamemBERT**  
  Exploration des embeddings contextualisés de tokens et des méthodes permettant de construire un embedding de phrase.

- **Partie B — Sentence-BERT**  
  Génération et comparaison d'embeddings de phrases optimisés pour la similarité sémantique.

- **Partie COURS — Exemples du cours**  
  Codes présentés dans les diapositives, disponibles pour être exécutés, modifiés et expérimentés.

**Organisation du travail :** les parties A et B constituent les activités du TP. La section 0 et la partie COURS sont des ressources complémentaires, à consulter selon les besoins.

Certaines cellules de code sont juste à tester, parfois en faisant varier certains paramètres, d'autres sont à compléter.  
Répondez aussi à plusieurs questions 💭 dans les cellules de texte.


In [ ]:
# (Optionnel) Installation
# !pip install transformers torch sentencepiece sentence-transformers scikit-learn matplotlib pandas

In [ ]:
# Importation des librairies principales
import torch                       # Librairie PyTorch : calcul tensoriel et deep learning
import numpy as np                 # Librairie NumPy : calcul numérique, manipulation de tableaux
import pandas as pd                # Librairie Pandas : manipulation de données tabulaires (DataFrame)
import matplotlib.pyplot as plt    # Librairie Matplotlib : visualisation de données (graphiques)

# Importation de classes et fonctions utiles pour l’analyse
from sklearn.decomposition import PCA                # Pour la réduction de dimension (visualiser en 2D/3D les embeddings)
from sklearn.metrics.pairwise import cosine_similarity  # Pour calculer la similarité cosinus entre vecteurs

# Importation de modèles de traitement du langage naturel (NLP)
from transformers import CamembertTokenizer, CamembertModel  # Modèle CamemBERT (transformer pré-entraîné pour le français)
from sentence_transformers import SentenceTransformer         # Version simplifiée pour produire directement des embeddings

# Fixation d'une graine aléatoire pour assurer la reproductibilité des résultats
np.random.seed(42)            # Graine pour les fonctions aléatoires de NumPy
torch.manual_seed(42)         # Graine pour les fonctions aléatoires de PyTorch

# Détection automatique du périphérique de calcul disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
# → Si une carte graphique NVIDIA compatible CUDA est détectée, on utilise "cuda" (GPU)
# → Sinon, on reste sur le "cpu" (processeur classique)

# Vérification du périphérique utilisé
device




## 0 — Rappels techniques : NumPy et PyTorch

Dans ce TP, les **embeddings** sont des vecteurs de nombres. Les représentations de plusieurs tokens sont regroupées dans des matrices.

Nous utiliserons deux types d'objets :

| | NumPy | PyTorch |
|---|---|---|
| Type | `ndarray` | `Tensor` |
| Usage | Calcul numérique et manipulation de données | Réseaux de neurones et calcul des embeddings |
| Exécution | CPU | CPU ou GPU |
| Calcul des gradients | Non | Possible, notamment pendant l'apprentissage |

**Exemple :** une phrase de 9 tokens, représentés par des vecteurs de 768 dimensions, correspond à une matrice de forme `(9, 768)`.

### Quelques opérations utiles

- `m.shape` : connaître les dimensions de la matrice.
- `m[i]` : sélectionner le vecteur de la ligne `i` (indices à partir de 0).
- `m[:, :10]` : sélectionner les 10 premières dimensions de tous les vecteurs.
- `m.T` : transposer la matrice (inverser lignes et colonnes).

Ces opérations s'utilisent de manière similaire avec NumPy et PyTorch.


### Conversion entre PyTorch et NumPy

Les modèles de deep learning comme CamemBERT utilisent des **tenseurs PyTorch** pour effectuer leurs calculs. Pour analyser ou présenter les résultats, nous utiliserons également **NumPy** et **Pandas**.

On rencontrera fréquemment l'instruction suivante :

```python
matrice_np = matrice_torch.detach().cpu().numpy()
```

Elle effectue trois opérations successives :

1. **`detach()`** : détache le tenseur du graphe de calcul utilisé par PyTorch pour calculer les gradients lors de l'apprentissage. Les valeurs numériques ne changent pas.
2. **`cpu()`** : place le tenseur en mémoire CPU. C'est nécessaire avant sa conversion en NumPy s'il se trouve sur un GPU.
3. **`numpy()`** : convertit le tenseur PyTorch en tableau NumPy (`ndarray`), utilisable par les bibliothèques d'analyse de données.

**Pourquoi convertir en NumPy ?**

Dans ce TP, nous voulons parfois présenter les embeddings sous forme de tableaux avec **Pandas** :

```python
matrice_np = matrice_torch.detach().cpu().numpy()
df = pd.DataFrame(matrice_np)
display(df)
```

Pandas manipule principalement des tableaux NumPy. La conversion permet donc de passer des résultats du modèle à un format adapté à leur analyse et à leur présentation.

**Attention :** la conversion n'est pas indispensable pour tout affichage. On peut afficher directement un tenseur PyTorch avec `print()`, et Pandas accepte également certains tenseurs CPU. Mais la conversion explicite évite les problèmes liés au GPU ou au calcul des gradients.

**À retenir :**

- **PyTorch** : calcul des embeddings et exécution du modèle.
- **NumPy** : manipulation numérique des résultats.
- **Pandas** : présentation des résultats sous forme de tableaux.

On conserve donc les tenseurs PyTorch pendant les calculs et on les convertit en NumPy lorsque cela facilite leur exploitation.



In [ ]:

# Une matrice de 3 vecteurs de 4 dimensions
m = torch.tensor([
    [1., 2., 3., 4.],
    [5., 6., 7., 8.],
    [9., 10., 11., 12.]
])

print("Type :", type(m))
print("Dimensions :", m.shape)

# Sélection d'un vecteur
print("\nPremier vecteur :", m[0])

# Sélection des deux premières dimensions de chaque vecteur
print("\nDeux premières dimensions :")
print(m[:, :2])

# Transposition
print("\nMatrice transposée :")
print(m.T)

# Conversion en NumPy
m_np = m.detach().cpu().numpy()

print("\nType après conversion :", type(m_np))
print("Dimensions :", m_np.shape)


# Conversion du tenseur PyTorch en tableau NumPy
m_np = m.detach().cpu().numpy()

print("Type avant conversion :", type(m))
print("Type après conversion :", type(m_np))

# Présentation de la matrice avec Pandas
df = pd.DataFrame(
    m_np,
    index=["token1", "token2", "token3"],
    columns=["dim1", "dim2", "dim3", "dim4"]
)

display(df)

#Suppression d'une dimension avec squeeze
x = torch.tensor([[10, 20, 30]])

print("AVANT :", x.shape)
print(x)

y = x.squeeze(0)

print("\nAPRÈS :", y.shape)
print(y)
#on passe d'une matrice d'une ligne à un vecteur
# squeeze et unsqueeze modifient la forme du tenseur,
# sans modifier ses valeurs.
# Ces opérations sont utiles pour gérer la dimension batch
# et préparer les tenseurs aux calculs des modèles.

**Vérifiez votre compréhension :**

Une matrice d'embeddings de forme `(12, 768)` représente combien de tokens ? Combien de dimensions possède le vecteur associé à chaque token ?

## Partie A — CamemBERT (embeddings de tokens → embeddings de phrase)

### 1- Charger le modèle et générer les tokens 


In [ ]:
# Chargement du tokenizer associé au modèle CamemBERT
tok = CamembertTokenizer.from_pretrained("camembert-base")
# → Le tokenizer transforme le texte brut en une séquence d’IDs d’entrée (tokens)
#   correspondant au vocabulaire appris lors du pré-entraînement.
#   Exemple : "Le chat dort" → [5, 345, 872, 6, ...]
#   Ce tokenizer utilise SentencePiece, un algorithme de segmentation en sous-mots (subwords).

# Chargement du modèle CamemBERT pré-entraîné
camem = CamembertModel.from_pretrained("camembert-base").to(device).eval()
# → Télécharge et instancie le modèle de base CamemBERT.
# → .to(device) : déplace le modèle sur le périphérique choisi (CPU ou GPU).
# → .eval() : met le modèle en mode évaluation (désactive le dropout et la mise à jour des poids).
#   ⚠️ Ce mode est utilisé pour l’inférence, pas pour l’entraînement.

# Message de confirmation
print("CamemBERT prêt sur :", device)
# → Affiche quel périphérique (CPU ou GPU) est utilisé pour le modèle.

### 2- Définir le corpus
Choisir des phrases   
- Une paire **proche** (paraphrases)
- Une paire **éloignée**


In [ ]:
sent_proches = ["Le chat dort sur le canapé.", "Un félin se repose sur le sofa."]
sent_eloignees = ["La voiture roule vite sur l'autoroute.", "Le ciel est bleu et lumineux."]

# A COMPLETER : Modifiez/ajoutez vos propres paires si vous voulez tester d'autres cas.


### 3- Afficher les tokens et la matrice des embeddings de tokens initiale (entrée dans le modèle )

La matrice d'embeddings de token initiale est de taille [vocab_size, hidden_dim] = [32005, 768]. Elle contient une ligne par token du vocabulaire complet.  
Elle est donnée par model.embeddings.word_embeddings.weight.

In [ ]:

# générer les tokens de la première phrase de sent_proches
s = sent_proches[0]
tokens = tok(s, return_tensors="pt").to(device) #tok est une instance de la classe CamembertTokenizere 
print(tokens)

# Extraire les identifiants de tokens (tensor 1D)
ids = tokens["input_ids"].squeeze(0)  # shape [seq_len]
# Reconstruction des vrais tokens pour les colonnes
tokens_text = tok.convert_ids_to_tokens(ids.tolist())

camem.eval() # lancement en mode inférence (pas entrainement)
with torch.no_grad(): #pas de retropropagation (descente de gradient)
    # --------------------------------------------------------
    # 🔍 Extraction de la matrice d'embeddings de tokens initiale
    # --------------------------------------------------------
    first_embed = camem.get_input_embeddings().weight # torch.Size([vocab_size, 768])

print("Taille du vocabulaire :", camem.config.vocab_size)
print("Shape de la matrice d'embeddings de token initiale (complète):", first_embed.shape)

# Récupérer leurs embeddings depuis la matrice initiale 
emb_init = first_embed[ids]  # shape [seq_len, 768]
print("Shape de la matrice d'embeddings de token initiale (tokens utiles) :", emb_init.shape)

# Conversion
emb_init_np = emb_init.detach().cpu().numpy() #ndarray [seq_len, hidden_dim]


# Construction d’un DataFrame : 10 premières dimensions × tokens
# Pour faciliter la lecture, on affiche uniquement les 10 premières
# dimensions des embeddings (sur les 768).

df = pd.DataFrame(
    np.round(emb_init_np[:, :10].T, 2),        # on garde 10 premières dimensions
    index=[f"dim{i+1}" for i in range(10)],    # lignes = dimensions
    columns=tokens_text                        # colonnes = tokens
)

print(f"\nEmbeddings initiaux pour la phrase : « {s} »")
display(df)

### 4- Visualiser la matrice d'embeddings de position (initiale)
 

In [ ]:
# --------------------------------------------------------
# 🔍 Extraction de la matrice d'embeddings de position
# --------------------------------------------------------
with torch.no_grad(): #pas de retropropagation (descente de gradient)
    # Matrice PyTorch des embeddings de position
    pos_embed = camem.embeddings.position_embeddings.weight
    # Copie NumPy uniquement pour l'affichage
    pos_embed_np = pos_embed.detach().cpu().numpy()
print("Shape de la matrice de position :", pos_embed.shape)

# Indices des positions réellement utilisés par CamemBERT
seq_len = tokens["input_ids"].shape[1]
pad = camem.config.pad_token_id or 1  # camembert-base: pad_token_id = 1
pos_ids = np.arange(seq_len) + pad + 1  # les positions réelles commencent à 2 car les deux premières lignes sont réservées
#et non utilsées ici

# Sous-matrice (10 premières dimensions pour visualisation)
pos_mat = np.round(pos_embed_np[pos_ids, :10], 3)  # [seq_len, 10]

# DataFrame : une ligne par token, une colonne par dimension
df_pos = pd.DataFrame(
    pos_mat,
    index=tokens_text,
    columns=[f"dim{j+1}" for j in range(10)]
)

print("\n🔹 Matrice des embeddings de position (10 premières dimensions) :")
display(df_pos)

### 5- Afficher la matrice d'embeddings de tokens (dernière couche)
Complétez le code suivant afin d'afficher la matrice d'embeddings de token initiale et finale (dernière couche) pour la deuxième phrase aussi.  


In [ ]:
#A COMPLETER
#Passage dans le modèle
with torch.no_grad():
    #indique que l'on est en inférence et non en entrainement -> pas de retropropagation
    out = camem(**tokens, output_hidden_states=True, return_dict=True)

#Récupération matrice dernier état
last_hidden = out.last_hidden_state.squeeze(0)  # [seq_len, 768]

# Affichage de la matrice d'embeddings (10 premières dimensions, tokens en colonnes)
mat = last_hidden.cpu().numpy()        # [seq_len, 768]
mat10 = np.round(mat[:, :10].T, 2)              # [10, seq_len]
dffin = pd.DataFrame(mat10, columns=tokens_text, index=[f"dim{i+1}" for i in range(10)])

print(f"\nEmbeddings initiaux pour la phrase : « {s} »")
display(df)
print(f"\nEmbeddings finaux pour la phrase : « {s} »")
display(dffin)


### 6- Comparer les matrices initiales (Tokens+Positions) et la matrice finale

Le code suivant permet de comparer les représentations initiales des tokens avec celles obtenues après leur passage dans le Transformer.

**Travail demandé :**

1. Complétez les instructions manquantes pour construire la matrice des représentations initiales à partir des matrices d'embeddings étudiées précédemment.
2. Exécutez le code et complétez les intitulés des deux tableaux pour l'affichage.
3. Expliquez pourquoi les représentations initiales et finales sont différentes.

Seules les 10 premières dimensions des vecteurs sont affichées.

In [ ]:
# A COMPLETER 
# Phrase étudiée
s = sent_proches[0]

# Tokenisation et préparation des identifiants
tokens = tok(s, return_tensors="pt").to(device)
ids = tokens["input_ids"].squeeze(0)
seq_len = len(ids)
tokens_text = tok.convert_ids_to_tokens(ids.tolist())

# Récupération des matrices d'embeddings
first_embed = camem.get_input_embeddings().weight
pos_embed = camem.embeddings.position_embeddings.weight

# Identifiants de position
pad = camem.config.pad_token_id
pos_ids = torch.arange(seq_len, device=device) + pad + 1

# Sélectionner les embeddings correspondant aux tokens de la phrase
word_used = ...  

# Sélectionner les embeddings correspondant à leurs positions
pos_used = ...

# Construire la représentation initiale
emb_sum = ...

# Calcul des représentations contextualisées
camem.eval()
with torch.no_grad():
    last_hidden = camem(**tokens).last_hidden_state.squeeze(0)

# Préparation des matrices pour l'affichage
emb_sum_np = emb_sum.detach().cpu().numpy()
matinit = np.round(emb_sum_np[:, :10].T, 2)

df = pd.DataFrame(
    matinit,
    columns=tokens_text,
    index=[f"dim{i+1}" for i in range(10)]
)

mat = last_hidden.detach().cpu().numpy()
mat10 = np.round(mat[:, :10].T, 2)

dffin = pd.DataFrame(
    mat10,
    columns=tokens_text,
    index=[f"dim{i+1}" for i in range(10)]
)

print(f"\nÀ compléter : « {s} »")
display(df)

print(f"\nÀ compléter : « {s} »")
display(dffin)

### 7- Evaluer l'évolution des embeddings : comparaison des vecteurs initiaux et finaux (similarité cosinus) 

Executez et étudiez le code suivant et répondez aux questions :   
  
    1️⃣ Que mesure la similarité cosinus ici ?  
    2️⃣ Interprétez les valeurs des similarités obtenues  ?  
    3️⃣ Qu’est-ce que cela montre sur la transformation effectuée par CamemBERT entre l’entrée et la sortie ?  
    4️⃣ Quels tokens changent le plus ? Pourquoi ?

In [ ]:

# Conversion CPU / NumPy
emb_sum_np = emb_sum.detach().cpu().numpy()
last_np = last_hidden.detach().cpu().numpy()

# Similarité cosinus token par token
cos_per_token = np.diag(cosine_similarity(emb_sum_np, last_np))

# Moyenne globale
print("Similarité moyenne entrée ↔ sortie :", round(np.mean(cos_per_token), 3))

# Détail par token
for token, cosval in zip(tokens_text, cos_per_token):
    print(f"{token:>10s}  →  cos={cosval:.3f}")

df_diff = pd.DataFrame({
    "token": tokens_text,
    "cos(entrée, sortie)": np.round(cos_per_token, 3)
})
display(df_diff)



### 8- Visualiser les **embeddings de tokens** en 2D (PCA)  

Complétez le code suivant pour visualiser les embeddings finaux des différentes phrases du corpus.


In [ ]:
# Fonction pour visualiser les embeddings de tokens (PCA 2D)
def plot_tokens(sentence, tok, camem, device, remove_special=True):
    """
    Visualise les embeddings contextualisés des tokens d'une phrase.

    La fonction récupère les vecteurs de la dernière couche de CamemBERT
    (768 dimensions), puis utilise une ACP (PCA) pour les projeter en 2D.

    Paramètres :
        sentence       : phrase à analyser
        tok            : tokenizer CamemBERT
        camem          : modèle CamemBERT préentraîné
        device         : périphérique de calcul (CPU ou GPU)
        remove_special : exclut les tokens spéciaux (<s>, </s>, <pad>)

    Chaque point du graphique représente un token de la phrase.
    Attention : les axes PCA ne correspondent pas à des dimensions
    sémantiques directement interprétables.
    """
    # Encodage
    enc = tok(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        out = camem(**enc, output_hidden_states=True, return_dict=True)

    # Matrice des tokens
    last_hidden = out.last_hidden_state.squeeze(0)  # [seq_len, 768]
    tokens = tok.convert_ids_to_tokens(enc["input_ids"].squeeze().tolist())

    # Filtrer les tokens spéciaux
    if remove_special:
        specials = {"<s>", "</s>", "<pad>"}
        keep = [t not in specials for t in tokens]
        tokens = [t for t,k in zip(tokens, keep) if k]
        last_hidden = last_hidden[[i for i,k in enumerate(keep) if k], :]

    # PCA 2D
    pts = PCA(n_components=2).fit_transform(last_hidden.detach().cpu().numpy())
    plt.figure(figsize=(6,5))
    plt.scatter(pts[:,0], pts[:,1])
    for i, t in enumerate(tokens):
        plt.text(pts[i,0]+0.01, pts[i,1], t, fontsize=9)
    plt.title(f"CamemBERT — Tokens (PCA 2D)\nPhrase: {sentence}")
    plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()
    
# A COMPLETER
# 🔎 Exemple : visualiser la 2 phrases  (proche)


# 🔎 Exemple : visualiser les deux  phrases éloignées


### 9- Implémenter le **Mean pooling** et extraire **CLS**
Le code suivant affiche les 10 premières composantes du vecteur d'embedding globale  de la phrase "Le chat dort sur le canapé.".  
Il montre que ce vecteur peut être calculé selon deux méthodes :    
- Mean = moyenne des embeddings des tokens pondérée par `attention_mask`
- CLS = vecteur du premier tokenrécupéré dans `<s>` : `last_hidden[0]`

Complétez le code suivant pour
- Générer et afficher une troisième version de l'embedding de la phrase en utilisant "Pooler output : CLS projeté par une couche supplémentaire"  


In [ ]:

def camembert_sentence_embedding(sentence: str, method="mean"):
    #retourne un sentence embedding correspondant à la phrase sentence en utilisant method
    e = tok(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        o = camem(**e, output_hidden_states=True, return_dict=True)
    lh = o.last_hidden_state.squeeze(0)
    if method == "cls":
        v = lh[0]
    else:
        if method == "mean" :
            m = e["attention_mask"].squeeze(0)
            v = (lh * m.unsqueeze(-1)).sum(0) / m.sum()
    return v.cpu().numpy()

pd.DataFrame({
    "Dimension":[f"dim{i+1}" for i in range(10)],
    "CLS": np.round(camembert_sentence_embedding(sent_proches[0], "cls")[:10],4),
    "Mean": np.round(camembert_sentence_embedding(sent_proches[0], "mean")[:10],4),
 })


### 10-  Calculer la similarité Cosinus pour deux phrases en testant plusieurs méthodes de calcul d'mbeddings

Le code suivant calcul les similarités entre plusieurs phrases en comparant les méthodes de calcul des embeddings de phrase.  
Exécutez ce code et interprétez les résultats en répondant aux questions suivantes    
    1️⃣ Quelle méthode (Mean, CLS ou Pooler) donne les similarités les plus élevées ?  
    2️⃣ Quelle méthode distingue le mieux les phrases proches et éloignées ?  
    3️⃣ Quelle méthode vous semble la plus adaptée pour mesurer la proximité sémantique entre phrases ?


In [ ]:
def cosine(u, v):
    """Calcule la similarité cosinus entre deux vecteurs numpy"""
    return cosine_similarity([u], [v])[0,0] #les vecteurs u et v sont mis sous la forme de matrice 2D attendues par la fonction
    #renvoie une matrice 1x1

# calcul des embeddings par mean
v1 = camembert_sentence_embedding(sent_proches[0], "mean")
v2 = camembert_sentence_embedding(sent_proches[1], "mean")
v3 = camembert_sentence_embedding(sent_eloignees[0], "mean")
v4 = camembert_sentence_embedding(sent_eloignees[1], "mean")

# calcul des embeddings par cls
w1 = camembert_sentence_embedding(sent_proches[0], "cls")
w2 = camembert_sentence_embedding(sent_proches[1], "cls")
w3 = camembert_sentence_embedding(sent_eloignees[0], "cls")
w4 = camembert_sentence_embedding(sent_eloignees[1], "cls")

# calcul des embeddings par pooler output
z1 = camembert_sentence_embedding(sent_proches[0], "pooler_vec")
z2 = camembert_sentence_embedding(sent_proches[1], "pooler_vec")
z3 = camembert_sentence_embedding(sent_eloignees[0], "pooler_vec")
z4 = camembert_sentence_embedding(sent_eloignees[1], "pooler_vec")

print("CamemBERT (Mean) — proches :", round(cosine(v1, v2), 3))
print("CamemBERT (Mean) — éloignées :", round(cosine(v3, v4), 3))
print("CamemBERT (cls) — proches :", round(cosine(w1, w2), 3))
print("CamemBERT (cls) — éloignées :", round(cosine(w3, w4), 3))
print("CamemBERT (pooler_vec) — proches :", round(cosine(z1, z2), 3))
print("CamemBERT (pooler_vec) — éloignées :", round(cosine(z3, z4), 3))


## Partie B — Sentence‑BERT (embeddings de phrase optimisés)



Dans cette partie, vous allez découvrir comment le modèle Sentence-BERT (SBERT) prolonge les modèles de type BERT pour produire des représentations vectorielles de phrases directement comparables entre elles.
L’objectif est de comprendre pourquoi SBERT est plus efficace que CamemBERT pour mesurer la proximité de sens entre phrases et pour les tâches de recherche sémantique.

In [ ]:
# Chargement d'un modèle Sentence-BERT pré-entrainé
sbert = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)
print("SBERT chargé.")

### 1 -  Encoder les mêmes phrases (sent_proches et sent_eloignees) avec SentenceBERT et comparer avec CamemBERT
- Encodez `sent_proches` et `sent_eloignees` avec SentenceBert
- Calculez les similarités cosinus
- Comparez les valeurs obtenues avec celles obtenues  CamemBERT

In [ ]:
# A COMPLETER

### 2— Mini‑recherche (retrieval)
Il s'agit ici d'exécuter et analyser un code non commenté.  
Votre tâche consiste à observer les résultats, identifier les indicateurs utilisés pour évaluer la qualité d'un système de recherche sémantique
et déduire leurs formules de calcul à partir du comportement du code. 

Répondez aux questions suivantes : 

    1️⃣ Observez les valeurs de similarité affichées dans les Top-3 :  
        Que semblent-elles mesurer entre la requête et les phrases du corpus ?  
    2️⃣ Pour chaque requête, comparez les Top-3 obtenus par CamemBERT et SBERT :  
        Quel modèle semble proposer les résultats les plus pertinents ?  
    3️⃣ Les lignes 'CamemBERT P@3' et 'SBERT P@3' affichent une mesure entre 0 et 1 :  
        - Que représente selon vous cet indicateur ?  
        - Comment pourrait-on le calculer à partir des résultats ?  
    4️⃣ Pourquoi SBERT obtient-il des valeurs de P@3 plus élevées ?  
        Que cela vous apprend-il sur la différence entre CamemBERT et SBERT ?

In [ ]:
corpus = [
    "Le chat dort sur le canapé du salon.",
    "Le chien court dans le jardin.",
    "La banque centrale relève ses taux directeurs.",
    "La météo annonce de la pluie demain.",
    "L'équipe de France a gagné le match.",
    "Un félin se repose tranquillement.",
    "Le conducteur accélère sur l'autoroute A6.",
    "Les étudiants révisent pour les examens."
]

queries = [
    "Un félin qui fait la sieste",
    "Hausse des taux d'intérêt"
]

def embed_camembert_batch(texts):
    vecs = [camembert_sentence_embedding(t, "mean") for t in texts]
    return np.vstack(vecs)

def search_topk(Q, D, texts, k=3):
    sims = cosine_similarity(Q, D)[0]
    idx = np.argsort(-sims)[:k]
    return [(float(sims[i]), texts[i]) for i in idx]

D_camem = embed_camembert_batch(corpus)
D_sbert = sbert_embed(corpus)

for q in queries:
    q_cam = camembert_sentence_embedding(q, "mean")[None, :]
    q_sbt = sbert_embed([q])
    print("\nQuery:", q)
    print("CamemBERT Top-3:")
    for s, t in search_topk(q_cam, D_camem, corpus):
        print(f"  {s:.3f}  {t}")
    print("SBERT Top-3:")
    for s, t in search_topk(q_sbt, D_sbert, corpus):
        print(f"  {s:.3f}  {t}")

gold = {
    "Un félin qui fait la sieste": {0, 5},
    "Hausse des taux d'intérêt": {2},
}

def precision_at_k(results, gold_set, k=3):
    retrieved = [t for _, t in results[:k]]
    hits = sum(1 for t in retrieved if corpus.index(t) in gold_set)
    return hits / k

for q in queries:
    q_cam = camembert_sentence_embedding(q, "mean")[None, :]
    q_sbt = sbert_embed([q])
    res_cam = search_topk(q_cam, D_camem, corpus, k=3)
    res_sbt = search_topk(q_sbt, D_sbert, corpus, k=3)
    p3_cam = precision_at_k(res_cam, gold[q], k=3)
    p3_sbt = precision_at_k(res_sbt, gold[q], k=3)
    print(f"\nQuery: {q}")
    print("CamemBERT P@3:", p3_cam, " | SBERT P@3:", p3_sbt)


---
# Partie COURS  — Modèles d’embeddings contextuels


In [ ]:
# Importer le modèle et le tokenizer
tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
model = CamembertModel.from_pretrained("camembert-base")
model.eval() #place le modèle en mode évaluation.

# Encoder une phrase
sentence = "La batterie est cassée."
inputs = tokenizer(sentence, return_tensors="pt")

print("\n=== ENTRÉES DU MODÈLE ===")
print("Phrase :", sentence)
print("Tokens :", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Identifiants :", inputs["input_ids"][0].tolist())
print("Masque d'attention :", inputs["attention_mask"][0].tolist())
print("Dimensions :", tuple(inputs["input_ids"].shape))

# Passage dans CamemBERT
with torch.no_grad(): #Indique qu'il n'est pas nécessaire de calculer les gradients, puisqu'on utilise le modèle sans l'entraîner
    outputs = model(**inputs, return_dict=True)

# Embeddings contextualisés des tokens
last_hidden = outputs.last_hidden_state

print("\n=== SORTIE DU MODÈLE ===")
print("Dimensions :", tuple(last_hidden.shape))
print("           (batch, tokens, dimension des embeddings)")

print("\nVecteur du premier token (5 premières composantes) :")
print(last_hidden[0, 0, :5].numpy().round(3))



In [ ]:
# Même phrase, mais avec une longueur fixée à 10 tokens
sentence = "La batterie est cassée."

inputs = tokenizer(
    sentence,
    return_tensors="pt",
    padding="max_length",
    max_length=10
)

print("\n=== ENTRÉES DU MODÈLE AVEC PADDING ===")
print("Phrase :", sentence)
print("Tokens :", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Identifiants :", inputs["input_ids"][0].tolist())
print("Masque d'attention :", inputs["attention_mask"][0].tolist())
print("Dimensions :", tuple(inputs["input_ids"].shape))

# Passage dans CamemBERT
with torch.no_grad():
    outputs = model(**inputs, return_dict=True)

last_hidden = outputs.last_hidden_state

print("\n=== SORTIE DU MODÈLE ===")
print("Dimensions :", tuple(last_hidden.shape))
print("Tokens réels :", int(inputs["attention_mask"].sum()))


In [ ]:
# ==================================================
# TROIS MÉTHODES POUR OBTENIR UN EMBEDDING DE PHRASE
# ==================================================

# 1. CLS : vecteur du premier token <s>
cls_vec = last_hidden[:, 0, :]

print("\n=== 1. CLS : PREMIER TOKEN <s> ===")
print("Dimensions :", tuple(cls_vec.shape))
print("5 premières valeurs :", cls_vec[0, :5].numpy().round(3))


# 2. MEAN POOLING : moyenne des vecteurs hors padding
mask = inputs["attention_mask"].unsqueeze(-1)

mean_vec = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1)

print("\n=== 2. MEAN POOLING ===")
print("Dimensions :", tuple(mean_vec.shape))
print("Tokens pris en compte :", int(mask.sum()))
print("5 premières valeurs :", mean_vec[0, :5].numpy().round(3))


# 3. POOLER OUTPUT : transformation du vecteur <s>
print("\n=== 3. POOLER OUTPUT ===")

if outputs.pooler_output is not None:
    pooler_vec = outputs.pooler_output
    print("Dimensions :", tuple(pooler_vec.shape))
    print("5 premières valeurs :", pooler_vec[0, :5].numpy().round(3))
else:
    print("Pooler output non disponible pour ce modèle.")

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Charger un modèle multilingue de Sentence-BERT
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Deux phrases de sens proche
sentences = [
    "Le chat dort sur le canapé.",
    "Un félin se repose sur le sofa."
]

# Générer directement les embeddings de phrases
embeddings = model.encode(sentences)

print("=== EMBEDDINGS DE PHRASES ===")
print("Nombre de phrases :", len(sentences))
print("Dimensions :", embeddings.shape)  # (2, 384)

for i, phrase in enumerate(sentences):
    print(f"\nPhrase {i+1} : {phrase}")
    print("5 premières composantes :", embeddings[i, :5].round(3))

# Comparer les deux embeddings
sim = cosine_similarity([embeddings[0]], [embeddings[1]])[0, 0]
dist = np.linalg.norm(embeddings[0] - embeddings[1])

print("\n=== COMPARAISON ===")
print(f"Similarité cosinus : {sim:.3f}")
print(f"Distance euclidienne : {dist:.3f}")


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

sentences = [
    "Le chat dort sur le canapé.",
    "Un félin se repose sur le sofa.",
    "Le chien joue dans le jardin.",
    "Un animal court dans le parc.",
    "La batterie du téléphone est déchargée.",
    "Mon smartphone n'a plus d'énergie.",
    "Le train arrive à la gare.",
    "Un voyageur attend son train."
]

embeddings = model.encode(sentences)

pts = PCA(n_components=2).fit_transform(embeddings)

plt.figure(figsize=(9, 6))
plt.scatter(pts[:, 0], pts[:, 1])

for i, phrase in enumerate(sentences):
    plt.annotate(f"P{i+1}", (pts[i, 0], pts[i, 1]),
                 xytext=(5, 5), textcoords="offset points")

plt.xlabel("Composante principale 1")
plt.ylabel("Composante principale 2")
plt.title("Projection 2D des embeddings de phrases")
plt.grid(alpha=0.3)
plt.show()

for i, phrase in enumerate(sentences):
    print(f"P{i+1} : {phrase}")